# Task 1. Clone Repository và Khám Phá File

## Mục tiêu

Task này chuẩn bị dữ liệu đầu vào cho pipeline Code Property Graph (CPG). Nhóm clone repository mục tiêu `huggingface/transformers-pr-agent`, kiểm tra commit đang dùng, khảo sát cấu trúc thư mục và chọn phạm vi file Python sẽ đưa vào Parser Service.

Repository này khá lớn, gồm mã nguồn chính, test, ví dụ, benchmark, tài liệu và script phụ trợ. Vì mục tiêu của lab là phân tích mã nguồn chương trình, nhóm chọn `src/` làm phạm vi parse chính để dữ liệu gọn và ít nhiễu hơn.

## Clone Repository

Repository được clone shallow bằng `--depth 1` để lấy snapshot mới nhất mà không tải toàn bộ lịch sử Git. Nếu thư mục đã tồn tại, notebook không clone lại mà dùng bản local hiện có.

In [1]:
from pathlib import Path
import subprocess
from collections import Counter

repo_url = "https://github.com/huggingface/transformers-pr-agent.git"
repo_path = Path("../transformers-pr-agent")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_path)],
        check=True,
    )
    print("Cloned repository:", repo_path)
else:
    print("Repository already exists:", repo_path)

remote_url = subprocess.check_output(
    ["git", "-C", str(repo_path), "remote", "get-url", "origin"],
    text=True,
).strip()
commit_hash = subprocess.check_output(
    ["git", "-C", str(repo_path), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Remote:", remote_url)
print("Commit hash:", commit_hash)

Repository already exists: ..\transformers-pr-agent
Remote: https://github.com/huggingface/transformers-pr-agent.git
Commit hash: 458c957fa1e8851825cd799f5d030876f0644194


## Khám Phá Cấu Trúc Repository

Cell dưới đây in cây thư mục rút gọn ở mức cao. Các thư mục build, cache và `.git` được bỏ qua để phần hiển thị tập trung vào cấu trúc nguồn.

In [2]:
def show_tree(path: Path, prefix: str = "", max_depth: int = 2, depth: int = 0):
    if depth > max_depth:
        return

    exclude_dirs = {".git", "__pycache__", ".pytest_cache", "_build"}
    items = sorted(
        [p for p in path.iterdir() if p.name not in exclude_dirs],
        key=lambda p: (p.is_file(), p.name.lower()),
    )

    shown_items = items[:30]
    for index, item in enumerate(shown_items):
        connector = "└── " if index == len(shown_items) - 1 else "├── "
        print(prefix + connector + item.name)

        if item.is_dir():
            extension = "    " if index == len(shown_items) - 1 else "│   "
            show_tree(item, prefix + extension, max_depth, depth + 1)

show_tree(repo_path)

├── .ai
│   ├── skills
│   │   └── add-or-fix-type-checking
│   └── AGENTS.md
├── .circleci
│   ├── config.yml
│   ├── create_circleci_config.py
│   ├── parse_test_outputs.py
│   └── TROUBLESHOOT.md
├── .github
│   ├── conda
│   │   ├── build.sh
│   │   └── meta.yaml
│   ├── ISSUE_TEMPLATE
│   │   ├── bug-report.yml
│   │   ├── config.yml
│   │   ├── feature-request.yml
│   │   ├── i18n.md
│   │   ├── migration.yml
│   │   └── new-model-addition.yml
│   ├── scripts
│   │   ├── assign_reviewers.py
│   │   └── codeowners_for_review_action
│   ├── workflows
│   │   ├── integration-failure-triage.yml
│   │   └── TROUBLESHOOT.md
│   ├── copilot-instructions.md
│   └── PULL_REQUEST_TEMPLATE.md
├── benchmark
│   ├── benches
│   │   └── llama.py
│   ├── config
│   │   └── generation.yaml
│   ├── utils
│   │   └── init_db.sql
│   ├── .gitignore
│   ├── __init__.py
│   ├── benchmark.py
│   ├── benchmarks_entrypoint.py
│   ├── default.yml
│   ├── grafana_dashboard.json
│   ├── grafana_datasource.

## Thống Kê File Python Theo Thư Mục

Trước khi chọn phạm vi phân tích, nhóm đếm toàn bộ file `.py` theo thư mục cấp cao nhất. Bước này giúp phân biệt mã nguồn chính với test, ví dụ, benchmark và script phụ trợ.

In [3]:
all_py_files = list(repo_path.rglob("*.py"))

by_top_level = Counter(
    file.relative_to(repo_path).parts[0]
    for file in all_py_files
)

print("Total Python files in repository:", len(all_py_files))
print("\nPython files by top-level path:")
for name, count in by_top_level.most_common():
    print(f"{name}: {count}")

Total Python files in repository: 4496

Python files by top-level path:
src: 2779
tests: 1516
examples: 96
utils: 75
docs: 11
benchmark_v2: 6
benchmark: 5
scripts: 3
.circleci: 2
conftest.py: 1
setup.py: 1
.github: 1


## Chọn Phạm Vi Phân Tích Chính

Nhóm chọn các file Python trong `src/` làm input chính cho Parser Service. Các file `setup.py`, `conftest.py` và thư mục cache không được đưa vào parse vì không đại diện cho mã nguồn thư viện cần phân tích.

In [4]:
source_root = repo_path / "src"
exclude_dirs = {".git", "__pycache__", ".pytest_cache"}
exclude_files = {"setup.py", "conftest.py"}

selected_py_files = [
    file for file in source_root.rglob("*.py")
    if not any(part in exclude_dirs for part in file.parts)
    and file.name not in exclude_files
]

summary = {
    "repository": repo_path.name,
    "remote": remote_url,
    "commit_hash": commit_hash,
    "source_root": source_root.as_posix(),
    "total_python_files": len(all_py_files),
    "tests_python_files": by_top_level.get("tests", 0),
    "selected_python_files": len(selected_py_files),
}

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nFirst 20 selected Python files:")
for file in selected_py_files[:20]:
    print(file.as_posix())

repository: transformers-pr-agent
remote: https://github.com/huggingface/transformers-pr-agent.git
commit_hash: 458c957fa1e8851825cd799f5d030876f0644194
source_root: ../transformers-pr-agent/src
total_python_files: 4496
tests_python_files: 1516
selected_python_files: 2779

First 20 selected Python files:
../transformers-pr-agent/src/transformers/activations.py
../transformers-pr-agent/src/transformers/audio_utils.py
../transformers-pr-agent/src/transformers/backbone_utils.py
../transformers-pr-agent/src/transformers/cache_utils.py
../transformers-pr-agent/src/transformers/configuration_utils.py
../transformers-pr-agent/src/transformers/conversion_mapping.py
../transformers-pr-agent/src/transformers/convert_slow_tokenizer.py
../transformers-pr-agent/src/transformers/convert_slow_tokenizers_checkpoints_to_fast.py
../transformers-pr-agent/src/transformers/core_model_loading.py
../transformers-pr-agent/src/transformers/debug_utils.py
../transformers-pr-agent/src/transformers/dependency_ver

## Kết Quả

Kết quả quan trọng của Task 1 là xác định được input ổn định cho các task sau:

- Repository mục tiêu: `huggingface/transformers-pr-agent`.
- Commit được ghi lại để kết quả parse có thể truy vết.
- Phạm vi parse chính: `transformers-pr-agent/src`.
- File test, ví dụ, benchmark, docs và script phụ trợ không nằm trong phạm vi parse chính.

Cách chọn này giúp Parser Service xử lý dữ liệu đại diện cho mã nguồn thư viện, đồng thời giảm dung lượng event khi chạy pipeline streaming.

## Reflection

Task 1 cho thấy bước khảo sát dữ liệu là cần thiết trước khi viết parser. Nếu parse toàn bộ repository, số lượng file phụ trợ và test sẽ làm dữ liệu CPG nhiều nhiễu hơn. Việc giới hạn vào `src/` giúp pipeline nhỏ gọn, dễ kiểm chứng và phù hợp với mục tiêu lab: chứng minh luồng xử lý incremental từ mã nguồn Python sang Kafka, Neo4j và MongoDB.